<a href="https://colab.research.google.com/github/busybee-123/Pollinator_Cam/blob/main/Train_and_evaluate_YOLO_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configuration

In [ ]:
import os

# Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

# Install ultralytics if not already installed
%pip install ultralytics

# Define the source path in Google Drive
source_dataset_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/synthetic_combined_dataset_v4'

# Define the base directory for the Pollinator Camera Project
project_base_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/'

## Load dataset into colab from Google Drive

In [ ]:
import shutil

# Define the destination path in Colab's local storage
local_dataset_path = '/content/dataset'

# Create the local directory if it doesn't exist
if not os.path.exists(local_dataset_path):
    os.makedirs(local_dataset_path)

# Copy the entire dataset directory to local Colab storage
try:
    # Using copytree will copy the directory and its contents recursively
    shutil.copytree(source_dataset_path, local_dataset_path, dirs_exist_ok=True)
    print(f"Successfully copied dataset from '{source_dataset_path}' to '{local_dataset_path}'")
except Exception as e:
    print(f"Error copying dataset: {e}")

# Update data_yaml_path to point to the local dataset
data_yaml_path = os.path.join(local_dataset_path, 'data.yaml')
print(f"Updated data.yaml path to: {data_yaml_path}")

## Train model and save .pt file to project folder in Google Drive

### Train with inverse frequency weighting

In [ ]:
import torch
from torch import nn

from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer


# Define the path to your data.yaml file
dataset_output_base_path = '/content/dataset'
data_yaml_path = os.path.join(dataset_output_base_path, 'data.yaml')

# Load a pre-trained YOLOv8n model (nano version, good for starting and quick training)
# 'v11' is not a standard release yet; YOLOv8 is the latest stable version from Ultralytics.
model = YOLO('yolo11m.pt')  # You can choose other versions like yolov8s.pt, yolov8m.pt, etc.

# Train the model
# Set parameters like epochs, image size, batch size.
# Augmentations are typically enabled by default in YOLOv8 training.
# You can fine-tune augmentation parameters in the data.yaml or via specific arguments if needed.
results = model.train(
    data=data_yaml_path,
    epochs=100,
    batch=-1,   # Batch size. Adjusted automatically based on your GPU memory.
    name='V10',
    shear=0.1,
    scale=0.5,
    perspective=0.0005,
    flipud=0.5,
    freeze=12,
    patience=5,
    lr0=0.005,
    cls_pw=1.0
)

print("\n--- YOLO model training complete! ---")
print(f"Results saved to: {model.trainer.save_dir}")

### Save best model to project directory

In [ ]:
import shutil
import os

# Source path: the directory where the training results are saved (from the previous cell)
source_dir = model.trainer.save_dir

# Assuming the best model is saved as 'best.pt' within the sav e_dir
source_model_path = os.path.join(source_dir, 'weights', 'best.pt')

# Destination path: the project base folder
destination_model_path = os.path.join(project_base_path, 'best.pt')

# Copy the model file
try:
    shutil.copy(source_model_path, destination_model_path)
    print(f"Successfully copied the trained model from '{source_model_path}' to '{destination_model_path}'")
except FileNotFoundError:
    print(f"Error: Model file not found at '{source_model_path}'")
except Exception as e:
    print(f"An error occurred while copying the model: {e}")

## View results

In [ ]:
import os
from IPython.display import Image, display

# Get the directory where training results are saved
results_dir = model.trainer.save_dir

print(f"Displaying plots from: {results_dir}")

# Define common plot filenames
plot_files = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'BoxF1_curve.png',
    'BoxPR_curve.png',
    'BoxP_curve.png',
    'BoxR_curve.png'
]

# Display each plot if it exists
for plot_file in plot_files:
    plot_path = os.path.join(results_dir, plot_file)
    if os.path.exists(plot_path):
        print(f"\n--- {plot_file.replace('_', ' ').replace('.png', '').upper()} ---")
        display(Image(filename=plot_path, width=800))
    else:
        print(f"Warning: {plot_file} not found at {plot_path}")

## Save results to Drive

In [ ]:
import shutil
import os
from datetime import datetime

# Get the directory where training results are saved (source)
source_results_dir = model.trainer.save_dir

# Create the 'YOLO results' folder if it doesn't exist
yolo_results_base_path = os.path.join(project_base_path, 'YOLO results')
os.makedirs(yolo_results_base_path, exist_ok=True)

# Create a subfolder name with the current date and time
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
destination_results_dir = os.path.join(yolo_results_base_path, timestamp)

# Copy the entire results directory
try:
    shutil.copytree(source_results_dir, destination_results_dir)
    print(f"Successfully copied all YOLO training results from '{source_results_dir}' to '{destination_results_dir}'")
except FileExistsError:
    print(f"Warning: Destination directory '{destination_results_dir}' already exists. Contents were not overwritten. If you wish to re-save, please delete the existing folder or choose a new timestamp.")
except Exception as e:
    print(f"An error occurred while copying the results: {e}")

## Evaluate model based on test set

In [ ]:
metrics = model.val(data=data_yaml_path,split='test')

### Display evaluation results

In [ ]:
import os
from IPython.display import Image, display

# Get the directory where training results are saved
results_dir = '/content/runs/detect/val'

print(f"Displaying plots from: {results_dir}")

# Define common plot filenames
plot_files = [
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'BoxF1_curve.png',
    'BoxPR_curve.png',
    'BoxP_curve.png',
    'BoxR_curve.png'
]

# Display each plot if it exists
for plot_file in plot_files:
    plot_path = os.path.join(results_dir, plot_file)
    if os.path.exists(plot_path):
        print(f"\n--- {plot_file.replace('_', ' ').replace('.png', '').upper()} ---")
        display(Image(filename=plot_path, width=800))
    else:
        print(f"Warning: {plot_file} not found at {plot_path}")

### Save to Drive

In [ ]:
import shutil
import os
from datetime import datetime

# Get the directory where training results are saved (source)
source_results_dir = '/content/runs/detect/val'

# Create the 'YOLO results' folder if it doesn't exist
yolo_results_base_path = os.path.join(project_base_path, 'YOLO results')
os.makedirs(yolo_results_base_path, exist_ok=True)

# Create a subfolder name with the current date and time
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
destination_results_dir = os.path.join(yolo_results_base_path,'test_val', timestamp)

# Copy the entire results directory
try:
    shutil.copytree(source_results_dir, destination_results_dir)
    print(f"Successfully copied all YOLO training results from '{source_results_dir}' to '{destination_results_dir}'")
except FileExistsError:
    print(f"Warning: Destination directory '{destination_results_dir}' already exists. Contents were not overwritten. If you wish to re-save, please delete the existing folder or choose a new timestamp.")
except Exception as e:
    print(f"An error occurred while copying the results: {e}")